# 05. Beyond the paper: the same simulated market with fat tails

**Terms used in this notebook.** Risk-free rate: the return on a one-month US Treasury bill. Excess return: a return minus the risk-free rate over the same month. Sharpe ratio: the average monthly excess return divided by the standard deviation of the excess return; the reward earned per unit of risk taken. Estimation window: the M months of past returns a rule sees when it forms its weights, 120 here. Rolling evaluation: moving the window forward one month at a time, forming the weights, and recording the return of the month after the window. Out-of-sample: measured on months the rule had not seen when it formed its weights. In-sample: measured on the same months that were used to form the weights. Estimation error: the difference between a mean or covariance estimated from a window and its true value. Tangency portfolio: the mix of risky assets with the highest Sharpe ratio when the true means and covariances are known. Critical window: the window length at which sample-based mean-variance starts to beat 1/N on average. Factor: a return series that moves many assets at once, for example the return of the whole market; the Fama-French factors SMB and HML are the returns of small firms over large ones and of cheap firms over expensive ones. Beta: how much an asset moves with the factor. Alpha: the part of an asset's expected return that its beta on the factor does not explain, zero in this simulation. Idiosyncratic: an asset's own noise, unrelated to the factor. Simulated history: the 24,000 months of returns the simulation produces. Seed: the number that fixes the random numbers a simulation uses; the same seed gives the same random numbers. Draw: one simulated history, produced from one seed. Standard error: the uncertainty of an estimate; for an average over M months it is the standard deviation of the months divided by the square root of M. Fat tails: extreme months more frequent and more extreme than a normal distribution allows. Kurtosis: a measure of how heavy the tails of a distribution are; excess kurtosis is zero for the normal distribution. Student-t: a fat-tailed relative of the normal distribution. Degrees of freedom: the Student-t's tail parameter, fewer meaning fatter tails. Scale mixture: a fat-tailed variable built by multiplying a normal one by a random scale. Crossing: in the simulation, the first window tested at which mean-variance's out-of-sample Sharpe ratio reaches 1/N's. Slack: the allowance of 0.01 within which a comparison still counts as in the expected direction. Bayes-Stein: the rule of Jorion (1986) that pulls each estimated mean towards one common value; this pulling is called shrinkage. Minimum variance: the rule that holds the fully invested portfolio with the lowest variance and uses no expected returns.

**What this notebook does.** The formula and the simulation of notebook 04 assume that returns are normally distributed and independent from month to month. The paper chose that setting because most portfolio rules are derived in it and it should favour mean-variance. This notebook reruns the Table 6 simulation with fat-tailed shocks and nothing else changed, and measures what the normality assumption is worth. The paper has no table for this, so the pre-committed statements are three expectations, written in `constants.py` before the run and each reported as a count of cells.

**Fat tails.** A normal distribution says a month more than three standard deviations below the mean happens about once in 370 months, once in a working life. On French's US market series from 1926 to 2026 (1,202 months), months more than three standard deviations below the mean occurred 11 times, against 1.6 expected under a normal distribution, about seven times more often. Fat tails means that extreme months are more frequent and more extreme than a normal distribution allows. Fat tails matter for estimation because the extreme months pull the sample mean and the sample covariance around: one crash month inside a 120-month window changes every covariance in the window and stays in it for ten years. For the same window, estimation error is larger under fat tails, and a rule that trusts its estimates should lose more.

**How the fat tails are built.** The Student-t distribution is the standard fat-tailed relative of the normal: the same bell shape in the middle, more weight in the tails, and one parameter, the degrees of freedom, that sets how much; the smaller it is, the fatter the tails. Two levels are run: 10 degrees of freedom, whose excess kurtosis of 1 is about that of a broad index's monthly returns, and 5, whose excess kurtosis of 6 is about that of an industry portfolio's. The construction is a scale mixture: each month's normal shocks, the same draws as notebook 04 from the same seed, are multiplied by one random scale factor for that month, drawn so that the variance is unchanged. Because one scale applies to every asset in the month, an extreme month hits all of them together, as crashes do. The means and covariances of the fat-tailed market are exactly those of notebook 04's, so the tangency portfolio, S*, S_ew and the formula's critical window are the same numbers, and only the rules' estimates of them change.

**What I expect to see, written before the run.** First, a construction check: 1/N and the tangency portfolio, which estimate nothing, keep their full-sample Sharpe ratios to within two standard errors (0.013). Second, the direction: in every cell of the Table 6 grid, every estimated rule's Sharpe ratio is no higher under fat tails than under normal shocks, allowing a slack of 0.01 for noise, and the loss grows as the tails fatten and as the window shortens. Third, the crossing: for 10 assets, the window at which mean-variance draws level with 1/N comes later under fat tails; for 25 and 50 assets, 1/N is so close to the tangency portfolio in this market that no crossing is expected within 12,000 months under any distribution, and the formula says so. Running time: three runs of the Table 6 grid, about six minutes, and the crossing table, about five.

## Modules

The modules of notebook 04; `simulate` gains the fat-tailed option and a kurtosis function.

In [ ]:
import os
os.makedirs("src/bp", exist_ok=True)
open("src/bp/__init__.py", "w").close()

In [ ]:
%%writefile src/bp/constants.py
"""
Pre-committed numbers for the study.

Every band, gate, sample boundary and design parameter is defined here,
one per line, with its source beside it. A notebook may read these names; it
may not introduce numbers of its own. The replication bands (section 3)
and the design of the extension (section 5) were fixed on 10 September 2026,
before any of the rules' code was written. Three entries were changed after that
date; each carries the date of the change, the evidence for it, and what it
replaced, so that a reader can judge whether the change was a specification
settled by the paper's own text or a number tuned to the results.

DGU throughout means DeMiguel, Garlappi and Uppal (2009), "Optimal Versus
Naive Diversification: How Inefficient is the 1/N Portfolio Strategy?",
Review of Financial Studies 22(5), 1915-1953. Page numbers are the journal's.
"""

# ---------------------------------------------------------------------------
# 1. Data: Ken French's library, downloaded at run time, never committed
# ---------------------------------------------------------------------------

FRENCH_BASE_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

# Every file the study reads. The vintage is whatever the library serves on the
# download date; the loader records the date, the SHA-256 of the zip and the
# CRSP vintage line printed at the top of each CSV.
FRENCH_FILES = {
    "factors3":   "F-F_Research_Data_Factors_CSV.zip",        # Mkt-RF, SMB, HML, RF, monthly from 1926-07
    "momentum":   "F-F_Momentum_Factor_CSV.zip",              # Mom, monthly from 1927-01
    "factors5":   "F-F_Research_Data_5_Factors_2x3_CSV.zip",  # Mkt-RF, SMB, HML, RMW, CMA, RF from 1963-07
    "ind10":      "10_Industry_Portfolios_CSV.zip",           # DGU "Industry" dataset
    "ind49":      "49_Industry_Portfolios_CSV.zip",           # the extension universe; carries firm counts and average size
    "size_bm25":  "25_Portfolios_5x5_CSV.zip",                # DGU "FF-1-factor" and "FF-4-factor" datasets
}

FRENCH_MISSING_CODES = (-99.99, -999.0)   # French's own convention, stated at the top of every portfolio file
FRENCH_RETURNS_ARE_PERCENT = True         # the loader divides by 100 once, at load time, and nowhere else

# ---------------------------------------------------------------------------
# 2. The DGU replication
# ---------------------------------------------------------------------------

DGU_SAMPLE_START = "1963-07"   # DGU Table 2, p. 1918, for every French-sourced dataset
DGU_SAMPLE_END = "2004-11"     # DGU Table 2, p. 1918
DGU_ESTIMATION_WINDOW = 120    # months; DGU section 2, p. 1927. M = 60 is in DGU's appendix and is out of scope.
DGU_RISK_AVERSION = 1.0        # gamma in the CEQ, DGU p. 1929: "the results we report are for the case of gamma = 1"
DGU_TCOST = 0.0050             # proportional cost per unit of one-way turnover; DGU p. 1929: 50 basis points, after Balduzzi and Lynch (1999)
TCOST_SENSITIVITY = 0.0100     # the one cost sensitivity reported: double the base cost
DGU_GMINC_LOWER_BOUND_FRACTION = 0.5   # g-min-c imposes w >= a * 1 with a = 1/(2N); DGU p. 1926
DGU_BS_COV_DOF = "M - N - 2"   # the Bayes-Stein covariance divides by M - N - 2; DGU equation (5), p. 1923

# Changed 22 September 2026: a specification the paper leaves ambiguous, settled
# by its own footnote and by replication. DGU's Lagrangian (8), p. 1925, shows
# the short-sale constraint alone, which with the normalisation of equation (1)
# would make gamma irrelevant. Footnote 22, p. 1934, says the constrained rules
# produce "corner solutions with all wealth invested in a single asset", which
# only happens when the budget constraint 1'w = 1 sits inside the optimisation
# with gamma = 1. Both readings were run: with the budget inside, mv-c on
# MKT/SMB/HML gives a Sharpe ratio of 0.1093 against the published 0.1084 and
# puts all wealth in one asset in 257 of 377 months; without it, 0.2382 and
# never. The other three datasets and the turnover and CEQ rows agree the same
# way. The reading that reproduces the paper's tables is used; the literal
# reading is kept as rules.mv_c_cone so the comparison can be rerun
# (notebook 03), and is not a reported rule.
DGU_CONSTRAINED_BUDGET_INSIDE = True

# The rules replicated: those whose only inputs are the window's
# sample moments (and, for Bayes-Stein, Jorion's prior built from them), plus
# the value-weighted market. g-min-c is min-c with the lower bound raised from
# 0 to 1/(2N) (DGU p. 1926). vw is included because its Sharpe ratio is the
# same 0.1138 in every French-sourced column of Table 3, which makes it a free
# check that the market series and the sample match the paper's.
DGU_RULES = ("ew", "mv", "bs", "min", "vw", "mv-c", "bs-c", "min-c", "g-min-c")
DGU_IN_SAMPLE_ROW = "mv (in sample)"   # DGU Table 3 row 2: mean-variance with M = T, the upper bound estimation error costs

# Excluded, each with the criterion. They are not built.
DGU_RULES_EXCLUDED = {
    "Bayesian diffuse-prior": "not reported by DGU (Table 1 note, p. 1917)",
    "dm": "Bayesian Data-and-Model (Pastor 2000); needs an asset-pricing prior, outside the moment-based set",
    "mp": "MacKinlay and Pastor (2000) missing-factor model; needs a maximum-likelihood factor structure, outside the moment-based set",
    "mv-min": "Kan and Zhou (2007) three-fund mixture of two rules already built; not part of the pre-specified set",
    "ew-min": "mixture of 1/N and minimum variance; not part of the pre-specified set",
    "multi-prior": "not reported by DGU (Table 1 note, p. 1917)",
}

# The datasets replicated: the four DGU built from French's site (DGU Table 2, p. 1918; Appendix A, pp. 1948-1949).
# Each is monthly excess returns over the T-bill rate from French's factor file.
DGU_DATASETS = {
    "Industry":     {"n_risky": 10, "factors": ("Mkt-RF",), "N": 11, "source": "ind10 + factors3"},
    "MKT/SMB/HML":  {"n_risky": 0,  "factors": ("Mkt-RF", "SMB", "HML"), "N": 3, "source": "factors3"},
    "FF-1-factor":  {"n_risky": 20, "factors": ("Mkt-RF",), "N": 21, "source": "size_bm25 + factors3"},
    "FF-4-factor":  {"n_risky": 20, "factors": ("Mkt-RF", "SMB", "HML", "Mom"), "N": 24, "source": "size_bm25 + factors3 + momentum"},
}
DGU_DATASETS_EXCLUDED = {
    "S&P Sectors": "Roberto Wessels' GICS sector data, not on French's site (DGU Appendix A.1)",
    "International": "MSCI country indices, not on French's site (DGU Appendix A.3)",
    "FF-3-factor": "DGU do not report it: 'very similar to FF-1-factor' (Table 3 note, p. 1931)",
}
DGU_EXCLUDE_LARGEST_SIZE_QUINTILE = True   # DGU footnote 24, p. 1949: the five largest-size portfolios of the 25 are dropped, leaving 20

# ---------------------------------------------------------------------------
# 3. Bands for the replication, written before the code ran
# ---------------------------------------------------------------------------

BAND_SHARPE_ABS = 0.03          # out-of-sample Sharpe ratio within 0.03 of DGU Table 3, per rule and dataset
CHECK_TOP3_RANKING = True        # the ranking of the top three rules within each dataset is preserved
BAND_TURNOVER_REL = 0.25        # turnover within 25% relative of DGU Table 5

# DGU Table 5 reports the unconstrained mean-variance rule's turnover at 606,594 times
# that of 1/N on the Industry dataset and 10,466 times on FF-1-factor. Those
# figures are dominated by the months in which the sample covariance is
# near-singular, and they move by orders of magnitude when French revises
# a single return. A 25% band on them tests the vintage and says nothing about
# the code. The band
# is therefore applied only where DGU's published relative turnover is below
# the value here; above it, the replicated figure is reported beside the
# published one with no pass or fail. Fixed 10 September 2026, before any
# of the rules' code existed.
BAND_TURNOVER_APPLIES_BELOW_RELATIVE = 100.0

# Changed 13 September 2026, after notebook 02 had run, on its evidence. The
# out-of-sample Sharpe band applies only to rules whose published relative
# turnover (DGU Table 5, panel A) is below the same level; a rule above it is
# reported beside the published figure with the range that ten basis points of
# noise produces, and carries no verdict. The level is the paper's own number
# and the turnover band already used it, so no new level enters here. The
# in-sample mean-variance row carries no verdict either: BAND_SHARPE_ABS was
# written for out-of-sample Sharpe ratios. Evidence: on the Industry, FF-1 and
# FF-4 datasets the unconstrained mean-variance rule moved by 0.12 to 0.25
# under ten basis points of noise, with the published figure inside the range,
# while the 1/N rows matched to the third decimal. Before the change the band
# applied to every out-of-sample row.
BAND_SHARPE_APPLIES_BELOW_RELATIVE = BAND_TURNOVER_APPLIES_BELOW_RELATIVE
BAND_SHARPE_DECISION_DATE = "2026-09-13"

# Sharpe ratios are compared on the same sample, 1963-07 to 2004-11, 120-month
# window, so the out-of-sample period is 1973-07 to 2004-11.
DGU_EXPECTED_T = 497           # months from 1963-07 to 2004-11 inclusive; a count, checked in notebook 01
DGU_EXPECTED_OOS = 377         # DGU_EXPECTED_T - DGU_ESTIMATION_WINDOW

# The replication extended to the latest month (notebook 07). Fixed 1 October
# 2026, before the notebook was written. The nine rules run through the same
# rolling evaluation on every month the vintage carries, from DGU_SAMPLE_START
# to its last month, and the measures are reported for three periods: the
# paper's out-of-sample period (1973-07 to 2004-11, the numbers of notebooks
# 02 and 03), the months since the paper (EXTENDED_NEW_PERIOD_START to the
# last month, every one of them out of sample for a window that begins 120
# months earlier), and the whole period. No published number exists for the
# new months, so the pre-committed statements are expectations, each reported
# as a count.
EXTENDED_NEW_PERIOD_START = "2004-12"   # the month after DGU's sample ends
# Expectations:
#  1. Count first: every dataset carries a return for every month from
#     DGU_SAMPLE_START to the vintage's last month, the four datasets end in
#     the same month, and the number of months is printed and recorded.
#  2. The paper's reading of Table 3 holds on the whole period: each
#     short-sale-constrained rule has a higher out-of-sample Sharpe ratio than
#     its unconstrained version on every dataset (mv-c against mv, bs-c
#     against bs, min-c against min; 12 comparisons).
#  3. Sample-based mean-variance is below 1/N on every dataset in the new
#     period and in the whole period (8 cells). The new months add about 260
#     to a problem that the formula of notebook 04 says needs thousands, so
#     nothing else is expected to change.
#  4. The number of cells in the new period in which a rule beats 1/N with a
#     Jobson-Korkie p-value below 0.05 is reported. On about 260 months the
#     standard error of a Sharpe-ratio difference is about 0.06, so few or
#     none are expected, and none would be claimed as a finding.
# Turnover inside a sub-period counts every rebalance that falls in it; the
# first out-of-sample month of the whole period has no rebalance, as in DGU.

# ---------------------------------------------------------------------------
# 4. DGU's analytical result and simulation (notebook 04)
# ---------------------------------------------------------------------------

# The analytical critical window (DGU Proposition 1, pp. 1937-1938; Figure 1, p. 1940).
# Panel E is the one calibrated to US stock-market data and the one the abstract quotes.
ANALYTIC_SHARPE_TANGENCY = 0.15    # S*, DGU Figure 1 panel E
ANALYTIC_SHARPE_1N = 0.12          # S_ew, DGU Figure 1 panel E
EXPECT_CRITICAL_M_LOWER = {25: 3000, 50: 6000}   # DGU p. 1941: "more than 3000 months" and "more than 6000 months"; the check is order of magnitude and direction in N

# The simulated dataset (DGU section 5.1, pp. 1941-1942).
SIM_RF_MEAN_ANNUAL = 0.02
SIM_RF_SD_ANNUAL = 0.02
SIM_FACTOR_MEAN_ANNUAL = 0.08
SIM_FACTOR_SD_ANNUAL = 0.16
SIM_ALPHA = 0.0
SIM_BETA_RANGE = (0.5, 1.5)          # loadings evenly spread
SIM_IDIO_VOL_RANGE = (0.10, 0.30)    # annual idiosyncratic volatility, uniform, cross-sectional mean 0.20
SIM_N = (10, 25, 50)
SIM_M = (120, 360, 6000)
SIM_T = 24000                        # months in one simulated history
SIM_SEED = 20260910                  # fixed so the run is reproducible; DGU's seed is unknown
# In one 24,000-month simulated history the standard error of a monthly Sharpe ratio is about
# 1/sqrt(24000) = 0.0065, so the comparison with DGU Table 6 uses the same band
# as the empirical tables, and in addition the sign of (mv minus 1/N) is checked
# against Table 6 in every (N, M) cell.
BAND_SIM_SHARPE_ABS = BAND_SHARPE_ABS
# Added 29 September 2026, when notebook 04 was written. Table 6 reports 1/N and
# "mv (true)" as constants across M within each N, so they are full-sample
# Sharpe ratios; the estimated rules are out of sample. The comparison follows
# that convention. As a second check, every rule is rerun on SIM_SPREAD_DRAWS
# alternative draws of the market (seeds 1 to SIM_SPREAD_DRAWS) and the
# published value is checked against the range they produce, the simulation's
# analogue of the fragility test in notebook 02.
SIM_SPREAD_DRAWS = 8

# Fat tails (notebook 05), an addition to the paper. Fixed 30 September 2026,
# before the notebook was written. The market above is rerun with every shock
# drawn from a Student-t distribution instead of a normal one, scaled so that
# the mean and covariance of returns are exactly those of the normal market:
# only the tails differ, and the tangency portfolio, S*, S_ew and the formula's
# critical window are the same numbers as in notebook 04. One scale draw per
# month is shared by the factor shock and every idiosyncratic shock (a
# multivariate t), so an extreme month hits all assets at once, which is how fat
# tails appear in markets. Months stay independent of one another; the
# normality assumption is relaxed and nothing else. The normal shocks are the
# same draws as notebook 04's (same seed), so each fat-tailed market is paired
# with its normal twin month by month.
SIM_T_DOF = (10, 5)                  # degrees of freedom. 10: excess kurtosis 1, about a broad index's monthly returns. 5: excess kurtosis 6, about an industry portfolio's. At 4 or below the fourth moment is infinite, the standard error of a Sharpe ratio is undefined, and no comparison could be read.
SIM_T_COMMON_SCALE = True            # one scale per month shared by all shocks (multivariate t); False would draw one scale per asset
# The crossing table: sample-based mean-variance against 1/N over a grid of
# windows, both measured on the same out-of-sample months, to locate the
# window at which the gap closes under each distribution. The formula's
# critical window for the market's own S* and S_ew is reported beside it.
SIM_CROSSING_WINDOWS = (120, 240, 360, 600, 1000, 2000, 3000, 4000, 6000, 9000, 12000)
# Expectations written before the run, each reported as a count of cells.
#  1. 1/N and mv (true), which use no estimates, have the same full-sample
#     Sharpe ratio under t shocks as under normal ones to within two standard
#     errors; a failure here would be a construction error.
#  2. Every estimated rule's Sharpe ratio in every (N, M) cell is no higher
#     under t shocks than under normal ones, beyond a slack of about one and a
#     half standard errors; the loss is expected to grow as the tails fatten
#     (5 against 10) and as the window shortens.
#  3. For N = 10 the crossing window is later under t shocks than under normal
#     ones. For N = 25 and 50, 1/N is so close to the tangency portfolio in
#     this market that no crossing is expected within 12,000 months under any
#     distribution, and the formula says so.
FAT_TAIL_MOMENT_CHECK_ABS = 0.013    # expectation 1: two standard errors of a 24,000-month Sharpe ratio
FAT_TAIL_DIRECTION_SLACK = 0.01      # expectation 2: a cell counts as in the expected direction if Sharpe(t) <= Sharpe(normal) + this

# Volatility that changes through time (notebook 06), the second addition to
# the paper. Fixed 1 October 2026, before the notebook was written. The market
# of notebook 04 is rerun with shocks whose volatility follows a GARCH(1,1)
# process (Bollerslev 1986), the standard model of volatility clustering: a
# large shock this month raises the volatility of next month's shock, and the
# effect decays slowly. The factor shock and every asset's idiosyncratic shock
# get their own, independent volatility process with the same parameters, so
# both the level and the shape of the covariance matrix move through time:
# when the factor's volatility is high relative to the assets' own, the
# correlations between assets are high, which is what markets do in a crisis.
# The unconditional mean and covariance are exactly those of the normal market
# (omega = 1 - alpha - beta puts the unconditional variance of each scaled
# shock at one), so the tangency portfolio, S*, S_ew and the formula's
# critical window are unchanged. The normal shocks are the same draws as
# notebook 04's, so the two markets are paired month by month. Months are no
# longer independent; the tails stay those of a normal distribution except for
# what the clustering itself produces (excess kurtosis 6 alpha^2 / (1 - beta^2
# - 2 alpha beta - 3 alpha^2), about 0.8 for these parameters).
SIM_GARCH_ALPHA = 0.10               # weight of last month's squared shock; a textbook monthly value
SIM_GARCH_BETA = 0.85                # weight of last month's variance; alpha + beta = 0.95 is the persistence, a half-life of a volatility shock of about 13 months
SIM_GARCH_OWN_PROCESS_PER_ASSET = True   # the factor and each idiosyncratic shock follow their own process; False would scale all shocks by one common volatility, which leaves the shape of the covariance unchanged and the rules' weights with it
# Expectations written before the run, each reported as a count of cells.
#  1. 1/N and mv (true) keep their full-sample Sharpe ratios to within
#     FAT_TAIL_MOMENT_CHECK_ABS of the normal market's; a failure is a
#     construction error.
#  2. Every estimated rule's Sharpe ratio in every (N, M) cell is no higher
#     under volatility clustering than in the normal market, beyond
#     FAT_TAIL_DIRECTION_SLACK. Unlike notebook 05, the loss is expected to be
#     visible, and largest for the rules that depend on the shape of the
#     covariance (min, min-c, g-min-c), because a 120-month average of a
#     covariance that moves is wrong in a known direction for the month that
#     follows.
#  3. For N = 10 the crossing window is later than in the normal market; for
#     N = 25 and 50 no crossing within 12,000 months under either market.
# Descriptive checks reported beside them: the first-order autocorrelation of
# the factor's squared return is about alpha (1 - alpha beta - beta^2) /
# (1 - 2 alpha beta - beta^2) = 0.18 under clustering and zero in the normal
# market, and the average correlation between assets is higher in the months
# of highest factor volatility than in the months of lowest.

# ---------------------------------------------------------------------------
# 5. The extension: constrained index tracking on the 49 industries
# ---------------------------------------------------------------------------

EXT_UNIVERSE = "ind49"
EXT_ESTIMATION_WINDOW = 120        # the same window as the replication, so one convention runs through the repository
EXT_REBALANCE = "monthly"
# The extension sample starts at the first month in which all 49 industries carry
# a return (French codes the early gaps as -99.99). That start month is a count:
# on the 202607 vintage, read on 10 September 2026, Hlth is the last industry to
# fill and does so in 1969-07. Notebook 01 asserts it on every run, so a vintage
# that moves it fails loudly.
EXT_SAMPLE_START = "1969-07"
EXT_SAMPLE_END = None              # the latest month the vintage carries

# The benchmark (the index the portfolio tracks). Count-first check: the
# cap-weighted combination of the 49 industries, weights from firm count times
# average firm size, must track the market factor plus the risk-free rate within
# 50 basis points a year of tracking error over the full sample. If it does not,
# the derived weights are dropped and the benchmark is the market series.
CAPW_TE_MAX_ANNUAL = 0.0050

# Constraints. Fixed 10 September 2026.
ACTIVE_WEIGHT_BOUND = 0.02         # each untilted industry within 2 percentage points of its benchmark weight; an absolute cap does not fit 49 industries whose benchmark weights run from under 0.5% to over 10%
TURNOVER_CAP_MONTHLY_ONE_WAY = 0.02   # one-way turnover per rebalance, against drifted weights; 2% a month is about 24% a year, the enhanced-indexing range
TILT_INDUSTRIES = ("Coal", "Oil", "Util")   # French's column names, stripped of padding
TILT_MAX_SHARE_OF_BENCHMARK = 0.5  # each tilted industry held at no more than half its benchmark weight, reported as a 50% reduction relative to the benchmark; a tilted industry's bounds are [0, 0.5 * benchmark], and ACTIVE_WEIGHT_BOUND does not apply to it, because the two would conflict for any industry above 4% of the benchmark
LONG_ONLY = True

# Constraint sets, cumulative, reported in this order.
CONSTRAINT_SETS = {
    "C0": ("long_only",),
    "C1": ("long_only", "active_weight_bound"),
    "C2": ("long_only", "active_weight_bound", "turnover_cap"),
    "C3": ("long_only", "active_weight_bound", "turnover_cap", "tilt"),
}
# The optimiser also accepts an exclusion list (weight zero on named assets). It
# is built and tested for later use and is not one of the reported constraint
# sets.

# Covariance estimators. LW_TARGET is provisional until the shrinkage target
# used by the constrained-portfolio literature has been checked before notebook
# 10; if a different Ledoit-Wolf target is standard for constrained portfolios,
# the target follows it and this line records the change.
COV_ESTIMATORS = ("sample", "ledoit_wolf", "factor", "pca")
LW_TARGET = "constant_correlation"     # Ledoit and Wolf (2004a), "Honey, I Shrunk the Sample Covariance Matrix"
FACTOR_MODEL_FACTORS = ("Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom")   # the Fama-French five factors plus momentum; the estimator is B F B' plus a diagonal of residual variances, B estimated in the same estimation window
PCA_N_COMPONENTS = 5                   # five components plus a diagonal idiosyncratic term
PCA_PC1_MARKET_CORR_MIN = 0.95         # the first component's correlation with the cap-weighted market return
PCA_USE_COVARIANCE = True              # covariance, not correlation, of monthly excess returns
PCA_SIGN_RULE = "positive market loading"
PCA_WINDOWS = {                        # three pre-specified windows; loading stability is the finding and carries no pass or fail
    "to_2019":   (None, "2019-12"),
    "2020_2022": ("2020-01", "2022-12"),
    "from_2023": ("2023-01", None),
}

# Attribution: factor-based attribution of each constrained portfolio's
# realised active return against the factors above, contribution by factor
# plus residual, with the identity as the check.
ATTRIBUTION_FACTORS = FACTOR_MODEL_FACTORS
ATTRIBUTION_IDENTITY_MAX_ABS_ERROR = 1e-9        # contributions plus residual minus active return, per month, in return units

# Specification count, written down before the code runs and reported.
# Replication: one specification per rule and dataset. Extension:
EXT_SPEC_COUNT = len(COV_ESTIMATORS) * len(CONSTRAINT_SETS)   # 16 optimised specifications
EXT_COST_LEVELS = (DGU_TCOST, TCOST_SENSITIVITY)              # each reported net of cost at both levels; the cost is applied after the fact and does not enter the optimisation

# ---------------------------------------------------------------------------
# 6. Output conventions
# ---------------------------------------------------------------------------

OUTPUT_DIR = "outputs"
PROVENANCE_FILE = "outputs/provenance_french.json"   # download date, URL, SHA-256 and CRSP vintage line per file
TIMEZONE = "Europe/Amsterdam"

In [ ]:
%%writefile src/bp/dgu_published.py
"""
The published numbers of DeMiguel, Garlappi and Uppal (2009), Review of
Financial Studies 22(5), transcribed from Tables 3, 4, 5 and 6, for the four
datasets built from Ken French's library.

The comparison tables in the notebooks are built from this module, never
typed into a notebook. tests/test_dgu_published.py re-reads the numbers from
the journal PDF when it is present on the machine (papers/, not committed) and
fails on any digit that differs.

Monthly figures throughout. Sharpe ratios and CEQ returns are for the
out-of-sample period 1973-07 to 2004-11 with a 120-month window (Tables 3 and
4); "mv (in sample)" uses the full sample. Turnover (Table 5) is the 1/N
rule's absolute monthly turnover and every other rule's turnover
relative to it; return-loss is the extra monthly return a rule needs to
match 1/N's Sharpe ratio net of a 50 basis point proportional cost.
"""

DATASETS = ("Industry", "MKT/SMB/HML", "FF-1-factor", "FF-4-factor")

# Table 3, p. 1931: monthly Sharpe ratio, with the Jobson-Korkie p-value of the
# difference from 1/N in the second tuple (None where the paper prints none).
SHARPE = {
    "ew":              ((0.1353, 0.2240, 0.1623, 0.1753), None),
    "mv (in sample)":  ((0.2124, 0.2851, 0.5098, 0.5364), None),
    "mv":              ((0.0679, 0.2186, 0.0128, 0.1841), (0.17, 0.46, 0.02, 0.45)),
    "bs":              ((0.0719, 0.2536, 0.0138, 0.1791), (0.19, 0.25, 0.02, 0.48)),
    "dm":              ((0.0581, 0.0016, 0.0004, 0.2355), (0.14, 0.00, 0.01, 0.17)),
    "min":             ((0.1554, 0.2493, 0.2778, -0.0183), (0.30, 0.23, 0.01, 0.01)),
    "vw":              ((0.1138, 0.1138, 0.1138, 0.1138), (0.01, 0.00, 0.01, 0.00)),
    "mp":              ((0.0533, -0.0002, 0.1238, 0.1230), (0.04, 0.00, 0.08, 0.03)),
    "mv-c":            ((0.0678, 0.1084, 0.1977, 0.2024), (0.03, 0.02, 0.02, 0.27)),
    "bs-c":            ((0.0819, 0.1514, 0.1955, 0.2062), (0.06, 0.09, 0.03, 0.25)),
    "min-c":           ((0.1425, 0.2493, 0.1546, 0.3580), (0.41, 0.23, 0.35, 0.00)),
    "g-min-c":         ((0.1451, 0.2467, 0.1615, 0.3028), (0.31, 0.25, 0.47, 0.00)),
    "mv-min":          ((0.0772, 0.2546, -0.0079, 0.1757), (0.21, 0.22, 0.01, 0.50)),
    "ew-min":          ((0.1576, 0.2503, 0.2608, -0.0161), (0.21, 0.17, 0.00, 0.01)),
}

# Table 4, p. 1934: monthly certainty-equivalent return, gamma = 1, with p-values.
CEQ = {
    "ew":              ((0.0050, 0.0039, 0.0073, 0.0072), None),
    "mv (in sample)":  ((0.0106, 0.0047, 0.0300, 0.0304), None),
    "mv":              ((-0.7816, 0.0045, -2.7142, -0.0829), (0.00, 0.31, 0.00, 0.01)),
    "bs":              ((-0.3157, 0.0043, -0.6504, -0.0362), (0.00, 0.32, 0.00, 0.06)),
    "dm":              ((-0.0319, -0.0084, -0.0296, 0.0110), (0.01, 0.04, 0.00, 0.11)),
    "min":             ((0.0052, 0.0039, 0.0100, -0.0002), (0.45, 0.45, 0.12, 0.00)),
    "vw":              ((0.0042, 0.0042, 0.0042, 0.0042), (0.04, 0.44, 0.00, 0.00)),
    "mp":              ((0.0014, -0.0026, 0.0054, 0.0053), (0.05, 0.04, 0.09, 0.10)),
    "mv-c":            ((0.0023, 0.0030, 0.0090, 0.0075), (0.10, 0.28, 0.03, 0.42)),
    "bs-c":            ((0.0031, 0.0038, 0.0088, 0.0074), (0.15, 0.46, 0.05, 0.44)),
    "min-c":           ((0.0047, 0.0039, 0.0060, 0.0051), (0.40, 0.45, 0.12, 0.17)),
    "g-min-c":         ((0.0048, 0.0038, 0.0067, 0.0070), (0.41, 0.40, 0.17, 0.45)),
    "mv-min":          ((-0.2337, 0.0044, -0.0875, -0.0318), (0.00, 0.28, 0.00, 0.07)),
    "ew-min":          ((0.0052, 0.0039, 0.0093, -0.0002), (0.42, 0.43, 0.12, 0.00)),
}

# Table 5, p. 1935. TURNOVER_1N is absolute; TURNOVER_RELATIVE is each rule's
# turnover divided by 1/N's; RETURN_LOSS is panel B.
TURNOVER_1N = (0.0216, 0.0237, 0.0162, 0.0198)
TURNOVER_RELATIVE = {
    "mv":       (606594.36, 2.83, 10466.10, 3553.03),
    "bs":       (10621.23, 1.85, 11796.47, 3417.81),
    "dm":       (21744.35, 76.30, 918.40, 32.46),
    "min":      (21.65, 1.11, 45.47, 6.83),
    "vw":       (0.0, 0.0, 0.0, 0.0),
    "mp":       (11.98, 59.41, 2.39, 2.07),
    "mv-c":     (7.17, 4.12, 17.53, 13.82),
    "bs-c":     (7.22, 3.65, 17.32, 13.07),
    "min-c":    (2.58, 1.11, 3.93, 1.76),
    "g-min-c":  (1.52, 1.09, 1.78, 1.70),
    "mv-min":   (9927.09, 2.61, 4292.16, 4857.19),
    "ew-min":   (15.66, 1.11, 34.10, 6.80),
}
RETURN_LOSS = {
    "mv":       (231.8504, 0.0003, 7.4030, 1.5740),
    "bs":       (9.4602, -0.0004, 2.0858, 1.1876),
    "dm":       (8.9987, 0.0393, 0.1302, -0.0007),
    "min":      (0.0015, -0.0004, -0.0008, 0.0024),
    "vw":       (0.0037, 0.0157, 0.0021, 0.0028),
    "mp":       (0.0050, 0.0227, 0.0023, 0.0030),
    "mv-c":     (0.0048, 0.0041, -0.0005, 0.0002),
    "bs-c":     (0.0038, 0.0023, -0.0004, -0.0000),
    "min-c":    (-0.0001, -0.0004, 0.0006, -0.0025),
    "g-min-c":  (-0.0003, -0.0003, 0.0001, -0.0029),
    "mv-min":   (6.8115, -0.0003, 0.9306, 1.8979),
    "ew-min":   (0.0008, -0.0004, -0.0011, 0.0024),
}

# Table 6, p. 1943: simulated data, monthly Sharpe ratio, columns ordered
# (N=10: M=120, 360, 6000), (N=25: same), (N=50: same).
SIM_COLUMNS = tuple((n, m) for n in (10, 25, 50) for m in (120, 360, 6000))
SIM_SHARPE = {
    "ew":        (0.1356, 0.1356, 0.1356, 0.1447, 0.1447, 0.1447, 0.1466, 0.1466, 0.1466),
    "mv (true)": (0.1477, 0.1477, 0.1477, 0.1477, 0.1477, 0.1477, 0.1477, 0.1477, 0.1477),
    "mv":        (-0.0019, 0.0077, 0.1416, 0.0027, 0.0059, 0.1353, 0.0078, -0.0030, 0.1212),
    "bs":        (-0.0021, 0.0087, 0.1416, 0.0031, 0.0074, 0.1363, 0.0076, -0.0035, 0.1229),
    "dm":        (0.0725, 0.1475, 0.1464, 0.0133, 0.1473, 0.1457, 0.0201, 0.0380, 0.1430),
    "min":       (0.1113, 0.1181, 0.1208, 0.0804, 0.0911, 0.0956, 0.0491, 0.0676, 0.0696),
    "mp":        (0.1171, 0.1349, 0.1354, 0.1265, 0.1442, 0.1446, 0.1312, 0.1460, 0.1465),
    "mv-c":      (0.0970, 0.1121, 0.1276, 0.1011, 0.1150, 0.1315, 0.1111, 0.1194, 0.1355),
    "bs-c":      (0.1039, 0.1221, 0.1317, 0.1095, 0.1222, 0.1350, 0.1162, 0.1251, 0.1381),
    "min-c":     (0.1284, 0.1324, 0.1335, 0.1181, 0.1227, 0.1248, 0.1224, 0.1277, 0.1292),
    "g-min-c":   (0.1289, 0.1312, 0.1320, 0.1311, 0.1336, 0.1348, 0.1364, 0.1402, 0.1415),
    "mv-min":    (-0.0029, 0.0106, 0.1414, 0.0087, 0.0172, 0.1361, 0.0016, -0.0068, 0.1229),
    "ew-min":    (0.1116, 0.1184, 0.1211, 0.0810, 0.0918, 0.0964, 0.0496, 0.0684, 0.0706),
}


def table(name: str):
    """A pandas DataFrame of one published table, rules by dataset."""
    import pandas as pd
    if name == "sharpe":
        return pd.DataFrame({k: v[0] for k, v in SHARPE.items()}, index=DATASETS).T
    if name == "ceq":
        return pd.DataFrame({k: v[0] for k, v in CEQ.items()}, index=DATASETS).T
    if name == "turnover_relative":
        return pd.DataFrame(TURNOVER_RELATIVE, index=DATASETS).T
    if name == "return_loss":
        return pd.DataFrame(RETURN_LOSS, index=DATASETS).T
    if name == "sim_sharpe":
        cols = pd.MultiIndex.from_tuples(SIM_COLUMNS, names=["N", "M"])
        return pd.DataFrame(SIM_SHARPE, index=cols).T
    raise KeyError(name)

In [ ]:
%%writefile src/bp/rules.py
"""
Portfolio rules from DeMiguel, Garlappi and Uppal (2009), section 1.

Each rule is a function of one estimation window of monthly excess returns,
an array of shape (M, N), and returns the vector of relative weights on the N
risky assets, DGU equation (1): the mean-variance solution x = Sigma^-1 mu / gamma
normalised by the absolute value of its sum, so that the weights sum to +1 or
to -1 and the direction of the position is preserved in the few windows where
the risky assets are net short.

Notation follows the paper: mu and Sigma are the sample moments of the window,
M its length, N the number of assets. The risk aversion gamma cancels in the
normalisation for every unconstrained rule, which is why it does not appear.

Rules in this module, with the DGU abbreviation and section:
  ew       1/N, section 1.1
  mv       sample-based mean-variance, section 1.2
  bs       Bayes-Stein shrinkage, section 1.3.2, Jorion (1986)
  min      minimum variance, section 1.4.1
  vw       the value-weighted market, section 1.4.2 (all weight on the market column)
  mv-c     short-sale-constrained mean-variance, section 1.5
  bs-c     short-sale-constrained Bayes-Stein, section 1.5
  min-c    short-sale-constrained minimum variance, section 1.5
  g-min-c  minimum variance with every weight at least 1/(2N), section 1.5

The constrained rules are quadratic programmes. Each is solved exactly with
the Lawson-Hanson non-negative least squares algorithm (scipy.optimize.nnls)
after a change of variables, so there is no solver setting to tune:
  minimise  x' Sigma x / 2 - mu' x   subject to x >= 0
is, with Sigma = L L' (Cholesky), the same as minimising |L' x - L^-1 mu|^2,
a non-negative least squares problem. A budget constraint 1'w = 1 is added
as one heavily weighted extra row, then the result is renormalised; a lower
bound w >= a is a shift of variables. tests/ checks every one against an
independent solver.
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import nnls

from . import constants as C


def sample_moments(R: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Sample mean and covariance (divisor M - 1) of an (M, N) window."""
    R = np.asarray(R, dtype=float)
    mu = R.mean(axis=0)
    Sigma = np.cov(R, rowvar=False, ddof=1)
    return mu, np.atleast_2d(Sigma)


def normalise(x: np.ndarray) -> np.ndarray:
    """DGU equation (1): w = x / |1'x|."""
    s = float(np.sum(x))
    if s == 0.0:
        raise ZeroDivisionError("portfolio weights sum to zero; cannot normalise")
    return x / abs(s)


def ew(R: np.ndarray) -> np.ndarray:
    n = np.asarray(R).shape[1]
    return np.full(n, 1.0 / n)


def mv(R: np.ndarray) -> np.ndarray:
    mu, Sigma = sample_moments(R)
    x = np.linalg.solve(Sigma, mu)
    return normalise(x)


def min_variance(R: np.ndarray) -> np.ndarray:
    _, Sigma = sample_moments(R)
    ones = np.ones(Sigma.shape[0])
    x = np.linalg.solve(Sigma, ones)
    return x / x.sum()


def vw(market_index: int):
    """The value-weighted market: weight one on the market column, zero elsewhere."""
    def rule(R: np.ndarray) -> np.ndarray:
        w = np.zeros(np.asarray(R).shape[1])
        w[market_index] = 1.0
        return w
    rule.__name__ = "vw"
    return rule


def in_sample_sharpe(R: np.ndarray, rule=mv) -> float:
    """DGU equation (13): the Sharpe ratio of a rule estimated once on the whole sample."""
    mu, Sigma = sample_moments(R)
    w = rule(R)
    return float(w @ mu / np.sqrt(w @ Sigma @ w))


# ---------------------------------------------------------------------------
# Bayes-Stein, DGU section 1.3.2, equations (4) and (5); Jorion (1986)
# ---------------------------------------------------------------------------

def bayes_stein_from_moments(mu: np.ndarray, Sigma_mnm2: np.ndarray, M: int) -> tuple[np.ndarray, np.ndarray, float]:
    """Jorion's predictive moments from a window's mean and its covariance with divisor M - N - 2.

    Returns (mu_bs, Sigma_bs, phi). The mean is shrunk toward the mean return
    of the minimum-variance portfolio by phi (DGU equation 5); the covariance
    is inflated for the uncertainty in the mean, Jorion's predictive variance.
    """
    N = len(mu)
    Sigma_inv_1 = np.linalg.solve(Sigma_mnm2, np.ones(N))
    w_min = Sigma_inv_1 / Sigma_inv_1.sum()
    mu_min = float(mu @ w_min)
    d = mu - mu_min
    dist = float(d @ np.linalg.solve(Sigma_mnm2, d))     # (mu - mu_min 1)' Sigma^-1 (mu - mu_min 1)
    lam = (N + 2) / dist
    phi = lam / (M + lam)                                 # equals (N+2) / ((N+2) + M * dist), DGU equation (5)
    mu_bs = (1.0 - phi) * mu + phi * mu_min
    Sigma_bs = Sigma_mnm2 * (1.0 + 1.0 / (M + lam)) + (lam / (M * (M + 1.0 + lam))) * np.outer(np.ones(N), np.ones(N)) / Sigma_inv_1.sum()
    return mu_bs, Sigma_bs, phi


def bayes_stein_moments(R: np.ndarray) -> tuple[np.ndarray, np.ndarray, float]:
    """The predictive mean and covariance of Jorion (1986), as DGU implement them, from a window of returns.

    The sample covariance here divides by M - N - 2 (DGU equation 5), the
    unbiased estimator of the inverse. Returns (mu_bs, Sigma_bs, phi).
    """
    R = np.asarray(R, dtype=float)
    M, N = R.shape
    mu = R.mean(axis=0)
    dev = R - mu
    Sigma = dev.T @ dev / (M - N - 2)
    return bayes_stein_from_moments(mu, Sigma, M)


def bs(R: np.ndarray) -> np.ndarray:
    mu_bs, Sigma_bs, _ = bayes_stein_moments(R)
    return normalise(np.linalg.solve(Sigma_bs, mu_bs))


# ---------------------------------------------------------------------------
# Constrained rules, DGU section 1.5, solved as non-negative least squares
# ---------------------------------------------------------------------------

_BUDGET_WEIGHT = 1e6   # weight of the budget row; the residual on 1'w = 1 is then below 1e-12 and the result is renormalised anyway


def _cholesky_upper(Sigma: np.ndarray) -> np.ndarray:
    """L' with Sigma = L L'. A tiny ridge keeps a near-singular window solvable."""
    N = Sigma.shape[0]
    try:
        L = np.linalg.cholesky(Sigma)
    except np.linalg.LinAlgError:
        L = np.linalg.cholesky(Sigma + 1e-12 * np.trace(Sigma) / N * np.eye(N))
    return L.T


def nonneg_mean_variance(mu: np.ndarray, Sigma: np.ndarray) -> np.ndarray:
    """argmax mu'x - x'Sigma x / 2 over x >= 0, as NNLS: minimise |L'x - L^-1 mu|^2.

    This is DGU's Lagrangian (8) read literally: the non-negativity constraint
    alone, the scale then removed by normalise(). It is kept so that the two
    readings of the equation can be compared (notebook 03); budget_mean_variance
    is the reading that reproduces the paper's tables.
    """
    Lt = _cholesky_upper(Sigma)
    b = np.linalg.solve(Lt.T, mu)           # L^-1 mu
    x, _ = nnls(Lt, b)
    return x


def budget_mean_variance(mu: np.ndarray, Sigma: np.ndarray, gamma: float = C.DGU_RISK_AVERSION) -> np.ndarray:
    """argmax mu'w - gamma w'Sigma w / 2 over w >= 0 and 1'w = 1.

    The budget constraint sits inside the optimisation, so gamma no longer
    cancels: with gamma = 1 and monthly moments the risk term is small against
    the differences in means, and the rule often puts all wealth in the asset
    with the highest estimated mean. That is the "corner solutions with all
    wealth invested in a single asset" of DGU footnote 22, and it is what
    reproduces their Tables 3 to 5 (constants.DGU_CONSTRAINED_BUDGET_INSIDE).
    As NNLS: minimise |sqrt(gamma) L'w - L^-1 mu / sqrt(gamma)|^2 with the
    budget as a heavily weighted extra row.
    """
    N = len(mu)
    Lt = _cholesky_upper(Sigma)
    A = np.vstack([np.sqrt(gamma) * Lt, _BUDGET_WEIGHT * np.ones(N)])
    b = np.concatenate([np.linalg.solve(Lt.T, mu) / np.sqrt(gamma), [_BUDGET_WEIGHT]])
    w, _ = nnls(A, b)
    return w / w.sum()


def constrained_min_variance(Sigma: np.ndarray, lower: float = 0.0) -> np.ndarray:
    """argmin w'Sigma w over w >= lower, 1'w = 1, as NNLS on v = w - lower.

    Objective |L'(v + lower 1)|^2, so the NNLS target is -L' lower 1; the budget
    1'v = 1 - N lower is appended as a heavily weighted row.
    """
    N = Sigma.shape[0]
    Lt = _cholesky_upper(Sigma)
    ones = np.ones(N)
    A = np.vstack([Lt, _BUDGET_WEIGHT * ones])
    b = np.concatenate([-Lt @ (lower * ones), [_BUDGET_WEIGHT * (1.0 - N * lower)]])
    v, _ = nnls(A, b)
    w = v + lower
    return w / w.sum()


def mv_c(R: np.ndarray) -> np.ndarray:
    mu, Sigma = sample_moments(R)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu, Sigma)
    return normalise(nonneg_mean_variance(mu, Sigma))


def bs_c(R: np.ndarray) -> np.ndarray:
    mu_bs, Sigma_bs, _ = bayes_stein_moments(R)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu_bs, Sigma_bs)
    return normalise(nonneg_mean_variance(mu_bs, Sigma_bs))


def mv_c_cone(R: np.ndarray) -> np.ndarray:
    """Equation (8) read literally, for the comparison in notebook 03; not a reported rule."""
    mu, Sigma = sample_moments(R)
    return normalise(nonneg_mean_variance(mu, Sigma))


def min_c(R: np.ndarray) -> np.ndarray:
    _, Sigma = sample_moments(R)
    return constrained_min_variance(Sigma, lower=0.0)


def g_min_c(R: np.ndarray) -> np.ndarray:
    """DGU p. 1926: minimum variance with w >= a 1, a = 1/(2N), halfway between min-c and 1/N."""
    _, Sigma = sample_moments(R)
    N = Sigma.shape[0]
    return constrained_min_variance(Sigma, lower=C.DGU_GMINC_LOWER_BOUND_FRACTION / N)


# ---------------------------------------------------------------------------
# The same rules from a window's moments, for the long simulated histories of
# notebook 04, where recomputing the covariance at every month would dominate
# the running time. Each takes (mu, Sigma, M) with Sigma the sample covariance
# with divisor M - 1, and returns the same weights as its window-based twin
# (tests/test_rules_backtest.py checks the equality).
# ---------------------------------------------------------------------------

def _to_mnm2(Sigma: np.ndarray, M: int) -> np.ndarray:
    """Rescale a divisor-(M-1) covariance to divisor M - N - 2, DGU equation (5)."""
    N = Sigma.shape[0]
    return Sigma * (M - 1) / (M - N - 2)


def ew_m(mu, Sigma, M):
    return np.full(len(mu), 1.0 / len(mu))


def mv_m(mu, Sigma, M):
    return normalise(np.linalg.solve(Sigma, mu))


def min_m(mu, Sigma, M):
    x = np.linalg.solve(Sigma, np.ones(len(mu)))
    return x / x.sum()


def bs_m(mu, Sigma, M):
    mu_bs, Sigma_bs, _ = bayes_stein_from_moments(mu, _to_mnm2(Sigma, M), M)
    return normalise(np.linalg.solve(Sigma_bs, mu_bs))


def mv_c_m(mu, Sigma, M):
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu, Sigma)
    return normalise(nonneg_mean_variance(mu, Sigma))


def bs_c_m(mu, Sigma, M):
    mu_bs, Sigma_bs, _ = bayes_stein_from_moments(mu, _to_mnm2(Sigma, M), M)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu_bs, Sigma_bs)
    return normalise(nonneg_mean_variance(mu_bs, Sigma_bs))


def min_c_m(mu, Sigma, M):
    return constrained_min_variance(Sigma, lower=0.0)


def g_min_c_m(mu, Sigma, M):
    return constrained_min_variance(Sigma, lower=C.DGU_GMINC_LOWER_BOUND_FRACTION / len(mu))


MOMENT_RULES = {
    "ew": ew_m,
    "mv": mv_m,
    "bs": bs_m,
    "min": min_m,
    "mv-c": mv_c_m,
    "bs-c": bs_c_m,
    "min-c": min_c_m,
    "g-min-c": g_min_c_m,
}


# The registry. vw needs the market column and is built per dataset with vw(index).
RULES = {
    "ew": ew,
    "mv": mv,
    "bs": bs,
    "min": min_variance,
    "mv-c": mv_c,
    "bs-c": bs_c,
    "min-c": min_c,
    "g-min-c": g_min_c,
}

In [ ]:
%%writefile src/bp/backtest.py
"""
Rolling out-of-sample evaluation and the three performance measures of
DeMiguel, Garlappi and Uppal (2009), section 2.

The rolling sample: with T months of data and an estimation window of M
months, the rule is estimated on months t-M+1 to t and its weights are held
through month t+1, for t = M, ..., T-1. That gives T - M out-of-sample
monthly returns per rule (DGU p. 1928).

Measures, all monthly:
  Sharpe ratio        mean of out-of-sample excess returns over their standard
                      deviation, equation (12); the p-value of the difference
                      from 1/N is the Jobson-Korkie (1981) statistic with the
                      Memmel (2003) correction, footnote 16.
  CEQ return          mean minus gamma/2 times variance, equation (14), with
                      the delta-method p-value of footnote 18.
  turnover            average over months of the sum of absolute trades needed
                      to move from the drifted weights to the new target,
                      equation (15).
  net-of-cost return  equation (16), with a proportional cost c per unit of
                      one-way turnover.
  return-loss         equation (17): the extra mean return a rule needs for its
                      net Sharpe ratio to equal 1/N's.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy import stats

from . import constants as C


@dataclass
class Backtest:
    name: str
    oos: pd.Series            # out-of-sample excess return, indexed by the month it was earned
    weights: pd.DataFrame     # target weights, indexed by the month they were formed for
    turnover: pd.Series       # one-way trade at each rebalance, equation (15) summand
    drifted: pd.DataFrame     # weights before each rebalance


def drift(w: np.ndarray, r: np.ndarray) -> np.ndarray:
    """Weights after one month of excess returns r, before rebalancing (DGU p. 1929, w_{t+}).

    Position j grows to w_j (1 + r_j) and wealth to 1 + w'r, the residual
    1 - sum(w) sitting in the riskless asset at zero excess return. Dividing by
    the sum of the risky positions would lose the sign of a net-short portfolio;
    dividing by wealth keeps it, as DGU's normalisation does on purpose.
    """
    return w * (1.0 + r) / (1.0 + w @ r)


def rolling(returns: pd.DataFrame, rule, window: int = C.DGU_ESTIMATION_WINDOW, name: str | None = None) -> Backtest:
    R = returns.to_numpy(dtype=float)
    T, N = R.shape
    if T <= window:
        raise ValueError(f"{T} months is not more than the {window}-month window")
    idx = returns.index
    n_oos = T - window
    oos = np.empty(n_oos)
    weights = np.empty((n_oos, N))
    drifted = np.full((n_oos, N), np.nan)
    turnover = np.full(n_oos, np.nan)
    prev = None
    for k in range(n_oos):
        t = window + k                       # the month whose return is earned out of sample
        w = np.asarray(rule(R[t - window:t]), dtype=float)
        weights[k] = w
        if prev is not None:
            drifted[k] = prev
            turnover[k] = np.abs(w - prev).sum()
        oos[k] = w @ R[t]
        prev = drift(w, R[t])
    months = idx[window:]
    cols = list(returns.columns)
    return Backtest(
        name=name or getattr(rule, "__name__", "rule"),
        oos=pd.Series(oos, index=months, name=name),
        weights=pd.DataFrame(weights, index=months, columns=cols),
        turnover=pd.Series(turnover, index=months, name="turnover"),
        drifted=pd.DataFrame(drifted, index=months, columns=cols),
    )


def rolling_moments(returns: pd.DataFrame | np.ndarray, rules: dict, window: int = C.DGU_ESTIMATION_WINDOW,
                    refresh_every: int = 1000, progress: int = 0) -> dict[str, Backtest]:
    """Rolling evaluation of a set of moment-based rules at once, for long simulated histories.

    The window's mean and covariance are kept as running sums, updated in
    O(N^2) per month (one row in, one row out) instead of recomputed from the
    whole window, and recomputed from scratch every refresh_every months so
    floating-point drift cannot accumulate. Every rule in `rules` is a
    function (mu, Sigma, M) -> weights, as in rules.MOMENT_RULES, and
    sees exactly the moments the window-based rules would compute (Sigma
    with divisor M - 1). Returns one Backtest per rule.
    """
    if isinstance(returns, pd.DataFrame):
        R = returns.to_numpy(dtype=float)
        idx = returns.index
        cols = list(returns.columns)
    else:
        R = np.asarray(returns, dtype=float)
        idx = pd.RangeIndex(len(R))
        cols = list(range(R.shape[1]))
    T, N = R.shape
    if T <= window:
        raise ValueError(f"{T} months is not more than the {window}-month window")
    n_oos = T - window
    names = list(rules)
    oos = {k: np.empty(n_oos) for k in names}
    weights = {k: np.empty((n_oos, N)) for k in names}
    drifted = {k: np.full((n_oos, N), np.nan) for k in names}
    turnover = {k: np.full(n_oos, np.nan) for k in names}
    prev = {k: None for k in names}
    s1 = R[:window].sum(axis=0)
    s2 = R[:window].T @ R[:window]
    for k_step in range(n_oos):
        t = window + k_step
        if k_step > 0 and k_step % refresh_every == 0:
            s1 = R[t - window:t].sum(axis=0)
            s2 = R[t - window:t].T @ R[t - window:t]
        mu = s1 / window
        Sigma = (s2 - window * np.outer(mu, mu)) / (window - 1)
        Sigma = 0.5 * (Sigma + Sigma.T)
        r_next = R[t]
        for k in names:
            w = np.asarray(rules[k](mu, Sigma, window), dtype=float)
            weights[k][k_step] = w
            if prev[k] is not None:
                drifted[k][k_step] = prev[k]
                turnover[k][k_step] = np.abs(w - prev[k]).sum()
            oos[k][k_step] = w @ r_next
            prev[k] = drift(w, r_next)
        # slide the window: drop month t - window, add month t
        r_out = R[t - window]
        s1 = s1 - r_out + r_next
        s2 = s2 - np.outer(r_out, r_out) + np.outer(r_next, r_next)
        if progress and (k_step + 1) % progress == 0:
            print(f"  {k_step + 1}/{n_oos} months", flush=True)
    months = idx[window:]
    return {k: Backtest(name=k, oos=pd.Series(oos[k], index=months, name=k),
                        weights=pd.DataFrame(weights[k], index=months, columns=cols),
                        turnover=pd.Series(turnover[k], index=months, name="turnover"),
                        drifted=pd.DataFrame(drifted[k], index=months, columns=cols)) for k in names}


def subperiod(bt: Backtest, start=None, end=None) -> Backtest:
    """The same rolling evaluation restricted to the out-of-sample months from start to end (Periods or
    'YYYY-MM' strings, inclusive). Every rebalance that falls inside the period counts towards its turnover."""
    idx = bt.oos.index
    lo = pd.Period(start, "M") if isinstance(start, str) else (start if start is not None else idx[0])
    hi = pd.Period(end, "M") if isinstance(end, str) else (end if end is not None else idx[-1])
    sl = slice(lo, hi)
    return Backtest(name=bt.name, oos=bt.oos.loc[sl], weights=bt.weights.loc[sl],
                    turnover=bt.turnover.loc[sl], drifted=bt.drifted.loc[sl])


# ---------------------------------------------------------------------------
# Measures
# ---------------------------------------------------------------------------

def sharpe(x: pd.Series) -> float:
    return float(x.mean() / x.std(ddof=1))


def ceq(x: pd.Series, gamma: float = C.DGU_RISK_AVERSION) -> float:
    return float(x.mean() - 0.5 * gamma * x.var(ddof=1))


def mean_turnover(bt: Backtest) -> float:
    """Equation (15): the average one-way trade per rebalance, over the rebalances that exist."""
    return float(bt.turnover.dropna().mean())


def net_of_cost(bt: Backtest, cost: float = C.DGU_TCOST) -> pd.Series:
    """Equation (16): (1 + r)(1 - c * turnover) - 1, with no trade charged in the first month."""
    to = bt.turnover.fillna(0.0)
    return (1.0 + bt.oos) * (1.0 - cost * to) - 1.0


def return_loss(bt: Backtest, benchmark: Backtest, cost: float = C.DGU_TCOST) -> float:
    """Equation (17), on net-of-cost returns."""
    k = net_of_cost(bt, cost)
    b = net_of_cost(benchmark, cost)
    return float(b.mean() / b.std(ddof=1) * k.std(ddof=1) - k.mean())


def sharpe_pvalue(x: pd.Series, y: pd.Series) -> float:
    """Jobson-Korkie (1981) test of equal Sharpe ratios with the Memmel (2003) correction, DGU footnote 16."""
    T = len(x)
    mi, mn = x.mean(), y.mean()
    si, sn = x.std(ddof=1), y.std(ddof=1)
    s_in = np.cov(x, y, ddof=1)[0, 1]
    theta = (2 * si**2 * sn**2 - 2 * si * sn * s_in
             + 0.5 * mi**2 * sn**2 + 0.5 * mn**2 * si**2
             - (mi * mn / (si * sn)) * s_in**2) / T
    z = (sn * mi - si * mn) / np.sqrt(theta)
    return float(2 * (1 - stats.norm.cdf(abs(z))))


def ceq_pvalue(x: pd.Series, y: pd.Series, gamma: float = C.DGU_RISK_AVERSION) -> float:
    """Delta-method test of equal CEQ returns, DGU footnote 18."""
    T = len(x)
    mi, mn = x.mean(), y.mean()
    vi, vn = x.var(ddof=1), y.var(ddof=1)
    v_in = np.cov(x, y, ddof=1)[0, 1]
    grad = np.array([1.0, -1.0, -0.5 * gamma, 0.5 * gamma])
    Theta = np.array([
        [vi, v_in, 0.0, 0.0],
        [v_in, vn, 0.0, 0.0],
        [0.0, 0.0, 2 * vi**2, 2 * v_in**2],
        [0.0, 0.0, 2 * v_in**2, 2 * vn**2],
    ])
    diff = (mi - 0.5 * gamma * vi) - (mn - 0.5 * gamma * vn)
    var = grad @ Theta @ grad / T
    z = diff / np.sqrt(var)
    return float(2 * (1 - stats.norm.cdf(abs(z))))


def evaluate(bts: dict[str, Backtest], benchmark: str = "ew", cost: float = C.DGU_TCOST) -> pd.DataFrame:
    """One row per rule: Sharpe, its p-value against the benchmark, CEQ, its p-value, turnover, relative turnover, return-loss."""
    b = bts[benchmark]
    rows = []
    for name, bt in bts.items():
        rows.append({
            "rule": name,
            "sharpe": sharpe(bt.oos),
            "sharpe_p": np.nan if name == benchmark else sharpe_pvalue(bt.oos, b.oos),
            "ceq": ceq(bt.oos),
            "ceq_p": np.nan if name == benchmark else ceq_pvalue(bt.oos, b.oos),
            "turnover": mean_turnover(bt),
            "turnover_rel": mean_turnover(bt) / mean_turnover(b),
            "return_loss": np.nan if name == benchmark else return_loss(bt, b, cost),
        })
    return pd.DataFrame(rows).set_index("rule")

In [ ]:
%%writefile src/bp/critical_window.py
"""
DGU Proposition 1 (pp. 1937-1938): how long an estimation window the
sample-based mean-variance rule needs before it beats 1/N on average.

The argument. An investor who knew the true mean vector mu
and covariance Sigma would hold the tangency portfolio and earn expected
utility S*^2 / (2 gamma), where S* is that portfolio's Sharpe ratio. Using
1/N instead costs a known amount, (S*^2 - S_ew^2) / (2 gamma), equation (B6),
where S_ew is the Sharpe ratio of the 1/N rule. Using the sample-based
rule costs an amount that depends on how large the estimation error is, which
shrinks as the window M lengthens and grows with the number of assets N,
equations (B1) to (B3). The critical window M* is the smallest M at which the
second cost falls below the first: from there on, estimating pays.

Three cases, from the proposition:
  1. mu unknown, Sigma known:      S*^2 - S_ew^2 - N/M > 0
  2. mu known, Sigma unknown:      k S*^2 - S_ew^2 > 0
  3. both unknown:                 k S*^2 - S_ew^2 - h > 0
with, equations (25) and (27),
  k = M/(M-N-2) * (2 - M(M-2)/((M-N-1)(M-N-4)))       (< 1)
  h = N M (M-2) / ((M-N-1)(M-N-2)(M-N-4))               (> 0)
The formulas need M > N + 4.
"""
from __future__ import annotations

import numpy as np


def k_factor(M: float, N: int) -> float:
    return M / (M - N - 2) * (2.0 - M * (M - 2) / ((M - N - 1) * (M - N - 4)))


def h_factor(M: float, N: int) -> float:
    return N * M * (M - 2) / ((M - N - 1) * (M - N - 2) * (M - N - 4))


def advantage(M: float, N: int, s_star: float, s_ew: float, case: int = 3) -> float:
    """Left-hand side of the proposition's inequality; positive means mean-variance wins on average."""
    if case == 1:
        return s_star**2 - s_ew**2 - N / M
    if case == 2:
        return k_factor(M, N) * s_star**2 - s_ew**2
    if case == 3:
        return k_factor(M, N) * s_star**2 - s_ew**2 - h_factor(M, N)
    raise ValueError(case)


def critical_window(N: int, s_star: float, s_ew: float, case: int = 3, m_max: int = 200_000) -> int | None:
    """The smallest window M (months) at which the sample-based mean-variance rule beats 1/N on average.

    None when no window up to m_max does, which happens when S_ew >= S* in case 1
    or whenever the 1/N Sharpe ratio is too close to the tangency one for the
    estimation cost ever to fall below the 1/N cost.
    """
    if case == 1:
        gap = s_star**2 - s_ew**2
        if gap <= 0:
            return None
        return int(np.floor(N / gap)) + 1
    for M in range(N + 5, m_max + 1):
        if advantage(M, N, s_star, s_ew, case) > 0:
            return M
    return None


# The six panels of DGU Figure 1 (p. 1940), with the values the text states for
# them (pp. 1939-1941). "about" and "more than" are the paper's own words.
FIGURE_1_PANELS = {
    "A": {"s_star": 0.40, "s_ew": 0.20, "stated": {25: (">", 200), 50: ("about", 600), 100: (">", 1200)}},
    "B": {"s_star": 0.40, "s_ew": 0.10, "stated": {25: ("=", 270), 50: ("=", 530), 100: ("=", 1060)}},
    "C": {"s_star": 0.20, "s_ew": 0.10, "stated": {25: ("about", 1000), 50: ("about", 2000)}},
    "D": {"s_star": 0.20, "s_ew": 0.05, "stated": {50: (">", 1500)}},
    "E": {"s_star": 0.15, "s_ew": 0.12, "stated": {25: (">", 3000), 50: (">", 6000)}},
    "F": {"s_star": 0.15, "s_ew": 0.08, "stated": {25: (">", 1600), 50: (">", 3200)}},
}


def figure_1_table() -> "pandas.DataFrame":
    """Every stated value beside the computed one, with a verdict in the paper's own terms."""
    import pandas as pd
    rows = []
    for panel, spec in FIGURE_1_PANELS.items():
        for N, (rel, value) in spec["stated"].items():
            m = critical_window(N, spec["s_star"], spec["s_ew"], case=3)
            if rel == ">":
                ok = m is not None and m > value
            elif rel == "about":
                ok = m is not None and abs(m - value) / value <= 0.15
            else:
                ok = m is not None and abs(m - value) <= 10
            rows.append({"panel": panel, "S*": spec["s_star"], "S_ew": spec["s_ew"], "N": N,
                         "stated": f"{rel} {value}", "computed": m, "verdict": "pass" if ok else "MISS"})
    return pd.DataFrame(rows).set_index(["panel", "N"])

In [ ]:
%%writefile src/bp/simulate.py
"""
DGU's simulated market, section 5.1 (pp. 1941-1942), with an optional
fat-tailed version that the paper does not run.

One factor. N risky assets: the factor portfolio itself plus N - 1 assets
whose excess returns follow R = alpha + beta R_f + e, with alpha = 0, betas
evenly spread between 0.5 and 1.5, and independent noise whose annual
volatility is drawn once per asset, uniformly between 10% and 30%, so the
cross-sectional average idiosyncratic volatility is 20%. The factor's excess
return is normal with annual mean 8% and standard deviation 16%. Everything
is monthly, i.i.d., normal. The paper draws T = 24,000 months once and runs
every rule through that one simulated history.

Because the true mean and covariance are known here, the tangency portfolio
can be held as a benchmark ("mv (true)" in Table 6). With alpha = 0 in a
one-factor model the tangency portfolio is the factor itself, so its Sharpe
ratio is the factor's, 0.08 / 0.16 a year, about 0.144 a month.

Fat tails (notebook 05). With dof set, every shock is multiplied by one
scale factor per month, sqrt((dof - 2) / W) with W chi-squared on dof degrees
of freedom, which turns the normal shocks into multivariate Student-t shocks
with the same mean and covariance: E[(dof - 2) / W] = 1. The normal shocks
are the same draws as the normal market's, so the two markets are paired
month by month and differ only in the tails.

Volatility clustering (notebook 06). With garch=True, each shock series (the
factor's and every asset's own) is multiplied by its own GARCH(1,1) scale,
s_t = sqrt(h_t) with h_t = omega + alpha (s_{t-1} z_{t-1})^2 + beta h_{t-1}
on the standardised shock z, and omega = 1 - alpha - beta so that the
unconditional variance of the scaled shock is one. A large shock raises next
month's volatility, and the rise decays at the rate alpha + beta. Because the
factor's volatility and each asset's own move separately, the correlations
between assets change through time: high when the factor is volatile, low
when it is quiet. The unconditional mean and covariance are unchanged.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from . import constants as C


@dataclass
class Market:
    N: int
    betas: np.ndarray          # length N, first entry 1.0 for the factor itself
    idio_var: np.ndarray       # monthly idiosyncratic variance per asset, 0 for the factor
    mu: np.ndarray             # true monthly excess means
    Sigma: np.ndarray          # true monthly covariance
    returns: np.ndarray        # (T, N) simulated monthly excess returns
    seed: int
    dof: float | None = None   # Student-t degrees of freedom of the shocks; None for normal
    garch: bool = False        # True when the shocks carry GARCH(1,1) volatility
    scales: np.ndarray | None = None   # (T, N) volatility multipliers applied to the shocks, when any

    @property
    def true_tangency_weights(self) -> np.ndarray:
        x = np.linalg.solve(self.Sigma, self.mu)
        return x / abs(x.sum())


def build_market(N: int, T: int = C.SIM_T, seed: int = C.SIM_SEED, dof: float | None = None,
                 garch: bool = False) -> Market:
    """One simulated history. The factor's returns depend on the seed only, so the
    three universes (N = 10, 25, 50) share it, as Table 6's identical "mv (true)"
    column across N implies; the idiosyncratic volatilities and shocks depend on
    the seed and on N. With dof set, the shocks are Student-t with that many
    degrees of freedom, scaled to the same covariance; the scale draws depend on
    the seed and on dof, and are shared across N like the factor. With garch
    set, each shock series carries its own GARCH(1,1) volatility. One relaxation
    at a time: dof and garch cannot both be set."""
    if dof is not None and dof <= 2:
        raise ValueError("dof must exceed 2 for the variance to exist")
    if dof is not None and garch:
        raise ValueError("one relaxation at a time: set dof or garch, not both")
    rng_factor = np.random.default_rng([seed, 0])
    rng = np.random.default_rng([seed, N])
    mu_f = C.SIM_FACTOR_MEAN_ANNUAL / 12.0
    sd_f = C.SIM_FACTOR_SD_ANNUAL / np.sqrt(12.0)
    lo, hi = C.SIM_BETA_RANGE
    betas = np.concatenate([[1.0], np.linspace(lo, hi, N - 1)])
    vlo, vhi = C.SIM_IDIO_VOL_RANGE
    idio_sd_annual = np.concatenate([[0.0], rng.uniform(vlo, vhi, size=N - 1)])
    idio_var = (idio_sd_annual / np.sqrt(12.0)) ** 2
    mu = C.SIM_ALPHA + betas * mu_f
    Sigma = sd_f**2 * np.outer(betas, betas) + np.diag(idio_var)
    f_shock = rng_factor.normal(0.0, sd_f, size=T)
    eps = rng.normal(0.0, np.sqrt(idio_var), size=(T, N))
    eps[:, 0] = 0.0
    scales = None
    if dof is not None:
        rng_scale = np.random.default_rng([seed, 1_000_000 + int(round(dof * 1000))])
        scale = np.sqrt((dof - 2.0) / rng_scale.chisquare(dof, size=T))
        f_shock = scale * f_shock
        eps = scale[:, None] * eps
        scales = np.repeat(scale[:, None], N, axis=1)
    if garch:
        # standardised shocks: the factor's in column 0 (the factor asset has no idiosyncratic shock),
        # each asset's own in columns 1 to N-1
        z = np.empty((T, N))
        z[:, 0] = f_shock / sd_f
        z[:, 1:] = eps[:, 1:] / np.sqrt(idio_var[1:])
        scales = garch_scales(z, C.SIM_GARCH_ALPHA, C.SIM_GARCH_BETA)
        f_shock = scales[:, 0] * f_shock
        eps[:, 1:] = scales[:, 1:] * eps[:, 1:]
    f = mu_f + f_shock
    returns = C.SIM_ALPHA + np.outer(f, betas) + eps
    return Market(N=N, betas=betas, idio_var=idio_var, mu=mu, Sigma=Sigma, returns=returns, seed=seed,
                  dof=dof, garch=garch, scales=scales)


def garch_scales(z: np.ndarray, alpha: float, beta: float) -> np.ndarray:
    """GARCH(1,1) volatility multipliers for standardised shocks z of shape (T, K), one process per column.

    h_t = omega + alpha (s_{t-1} z_{t-1})^2 + beta h_{t-1}, s_t = sqrt(h_t), with
    omega = 1 - alpha - beta so that the unconditional variance of s_t z_t is one;
    h_0 is set to that unconditional value.
    """
    if not 0 <= alpha and 0 <= beta and alpha + beta < 1:
        raise ValueError("need alpha, beta >= 0 and alpha + beta < 1")
    omega = 1.0 - alpha - beta
    T, K = z.shape
    h = np.empty((T, K))
    h[0] = 1.0
    for t in range(1, T):
        h[t] = omega + alpha * h[t - 1] * z[t - 1] ** 2 + beta * h[t - 1]
    return np.sqrt(h)


def autocorrelation(x: np.ndarray, lag: int = 1) -> float:
    """Sample autocorrelation of a series at the given lag."""
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    return float(np.sum(x[lag:] * x[:-lag]) / np.sum(x * x))


def excess_kurtosis(x: np.ndarray) -> float:
    """Sample kurtosis minus 3; zero for a normal distribution."""
    x = np.asarray(x, dtype=float)
    z = (x - x.mean()) / x.std(ddof=0)
    return float(np.mean(z**4) - 3.0)

## What fat tails look like

The factor's monthly return under the three distributions, from the same seed. Excess kurtosis is zero for a normal distribution, 1 for a t with 10 degrees of freedom and 6 for one with 5. The share of months beyond three standard deviations is 0.27% under a normal distribution. The last two columns check that the scaling left the mean and the variance where they were.

In [ ]:
import sys
sys.path.insert(0, "src")
import time
import numpy as np
import pandas as pd
from bp import constants as C
from bp import critical_window as CW
from bp import simulate as SIM
from bp import rules as S
from bp import backtest as B
from bp import dgu_published as P

pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

DISTS = [("normal", None)] + [(f"t, {d} d.o.f.", d) for d in C.SIM_T_DOF]
markets = {(N, label): SIM.build_market(N, dof=dof) for N in C.SIM_N for label, dof in DISTS}

rows = []
for label, dof in DISTS:
    f = markets[(10, label)].returns[:, 0]
    z = (f - f.mean()) / f.std(ddof=1)
    rows.append({"shocks": label, "excess kurtosis": SIM.excess_kurtosis(f), "months beyond 3 sd (%)": 100 * np.mean(np.abs(z) > 3),
                 "worst month": f.min(), "best month": f.max(),
                 "mean (true 0.0067)": f.mean(), "sd (true 0.0462)": f.std(ddof=1)})
pd.DataFrame(rows).set_index("shocks")

## Expectation 1: the rules that estimate nothing

1/N and the tangency portfolio held on the full 24,000 months, under each distribution and for each universe. They use no estimates, so the fat tails can only move them through sampling error, and the check is that they stay within two standard errors of their normal-shock values.

In [ ]:
rows = []
for N in C.SIM_N:
    ref = {}
    for label, dof in DISTS:
        mkt = markets[(N, label)]
        w = mkt.true_tangency_weights
        sr = {"ew": B.sharpe(pd.Series(mkt.returns.mean(axis=1))), "mv (true)": B.sharpe(pd.Series(mkt.returns @ w))}
        if dof is None:
            ref = sr
        for k, v in sr.items():
            rows.append({"N": N, "rule": k, "shocks": label, "Sharpe": v, "minus normal": v - ref[k],
                         "within check": bool(abs(v - ref[k]) <= C.FAT_TAIL_MOMENT_CHECK_ABS)})
check1 = pd.DataFrame(rows).set_index(["N", "rule", "shocks"])
print(f"expectation 1: {int(check1['within check'].sum())} of {len(check1)} within {C.FAT_TAIL_MOMENT_CHECK_ABS} of the normal-shock value")
check1

## Table 6 under each distribution

The nine cells of notebook 04, run three times: normal shocks (the same table as notebook 04), then 10 and 5 degrees of freedom. About two minutes per run.

In [ ]:
grid = {}
t_all = time.time()
for label, dof in DISTS:
    for N in C.SIM_N:
        mkt = markets[(N, label)]
        w_true = mkt.true_tangency_weights
        r_true, r_ew = mkt.returns @ w_true, mkt.returns.mean(axis=1)
        for M in C.SIM_M:
            bts = B.rolling_moments(mkt.returns, S.MOMENT_RULES, window=M)
            row = {"mv (true)": B.sharpe(pd.Series(r_true)), "ew": B.sharpe(pd.Series(r_ew))}
            row.update({k: B.sharpe(bt.oos) for k, bt in bts.items() if k != "ew"})
            grid[(label, N, M)] = row
    print(f"{label:14s} done, {time.time() - t_all:.0f}s so far")
tables = pd.DataFrame(grid).T
tables.index.names = ["shocks", "N", "M"]
for label, _ in DISTS:
    print(label)
    print(tables.loc[label].to_string())
    print()

## Expectation 2: every estimated rule, every cell, fat-tailed minus normal

A negative number is the direction expected. The count is over the seven estimated rules, nine cells and two fat-tailed distributions, 126 comparisons.

In [ ]:
estimated = [k for k in tables.columns if k not in ("ew", "mv (true)")]
normal = tables.loc["normal"]
diffs = {}
for label, dof in DISTS[1:]:
    d = (tables.loc[label] - normal)[estimated]
    diffs[label] = d
    in_dir = (d <= C.FAT_TAIL_DIRECTION_SLACK)
    print(f"{label}: {int(in_dir.values.sum())} of {in_dir.size} cells in the expected direction; mean change {d.values.mean():+.4f}; largest fall {d.values.min():+.4f}")
    print(d.round(4).to_string())
    print()

In [ ]:
print("mean change in Sharpe ratio by window, averaged over rules and N: the loss should be largest at M = 120 and shrink as the window grows")
pd.DataFrame({label: d.groupby(level="M").mean().mean(axis=1) for label, d in diffs.items()})

In [ ]:
print("mean change by rule, averaged over cells: which rules fat tails hurt most")
pd.DataFrame({label: d.mean(axis=0) for label, d in diffs.items()})

## Expectation 3: the crossing

Sample-based mean-variance against 1/N, both measured on the same out-of-sample months, over a grid of windows from 120 to 12,000 months. The gap is mean-variance's Sharpe ratio minus 1/N's; the crossing is the first window at which it is not negative. Beside it, the formula's critical window for this market's own S* and S_ew (notebook 04, case 3), which is the same number under every distribution because the moments are the same. About five minutes.

In [ ]:
pair = {k: S.MOMENT_RULES[k] for k in ("ew", "mv")}
gap_rows, cross_rows = [], []
t0 = time.time()
for N in C.SIM_N:
    mkt = markets[(N, "normal")]
    w = mkt.true_tangency_weights
    s_star = float(mkt.mu @ w / np.sqrt(w @ mkt.Sigma @ w))
    w_ew = np.full(N, 1.0 / N)
    s_ew = float(mkt.mu @ w_ew / np.sqrt(w_ew @ mkt.Sigma @ w_ew))
    formula = CW.critical_window(N, s_star, s_ew, case=3, m_max=C.SIM_T)
    for label, dof in DISTS:
        R = markets[(N, label)].returns
        gaps = {}
        for M in C.SIM_CROSSING_WINDOWS:
            bts = B.rolling_moments(R, pair, window=M)
            gaps[M] = B.sharpe(bts["mv"].oos) - B.sharpe(bts["ew"].oos)
            gap_rows.append({"N": N, "shocks": label, "M": M, "gap": gaps[M]})
        crossing = next((M for M in C.SIM_CROSSING_WINDOWS if gaps[M] >= 0), None)
        cross_rows.append({"N": N, "shocks": label, "S*": s_star, "S_ew": s_ew, "formula (months)": formula if formula is not None else f"none up to {C.SIM_T:,}",
                           "first window with mv >= 1/N": crossing if crossing is not None else f"none up to {C.SIM_CROSSING_WINDOWS[-1]:,}"})
    print(f"N={N:2d} done, {time.time() - t0:.0f}s so far")
crossing_table = pd.DataFrame(cross_rows).set_index(["N", "shocks"])
crossing_table

In [ ]:
print("the gap, mean-variance minus 1/N, by window: negative means 1/N is ahead")
pd.DataFrame(gap_rows).set_index(["N", "shocks", "M"])["gap"].unstack("M").round(4)

## What this notebook established, and what could be wrong

All three expectations are met, and the effect is small. Expectation 1: 18 of 18, so the fat-tailed markets are the same market with different tails. Expectation 2: 125 of 126 comparisons inside the slack, and the changes are small. With 5 degrees of freedom the estimated rules lose 0.002 in Sharpe ratio on average, 0.004 at the 120-month window and nothing at 6,000, and the largest single fall is 0.015; with 10 degrees of freedom the average change is +0.002, inside sampling error, so no effect is detectable. Expectation 3: for 10 assets the crossing moves from 3,000 months under normal shocks to 4,000 under both fat-tailed markets, against the formula's 4,536. The crossing is known only to the nearest window tested (the windows were 120, 240, 360, 600, 1,000, 2,000, 3,000, 4,000, 6,000, 9,000 and 12,000 months), and the gap between mean-variance and 1/N at 3,000 months under normal shocks is 0.0003, so the move is one step of that list and inside sampling error. For 25 and 50 assets no crossing occurs within 12,000 months under any distribution, as the formula says (17,753 months and none).

The effect is small for two reasons, one about the estimated means and one about the estimated covariances.

The means. Every rule that uses expected returns estimates them as the average return of each asset over the 120 months in the window. An average computed from 120 months is itself uncertain, because a different 120 months would give a different average. The size of that uncertainty is called the standard error, and it equals the standard deviation of the monthly returns divided by the square root of the number of months. For an asset whose monthly returns have a standard deviation of 5%, the standard error of its 120-month average is 5% divided by the square root of 120, about 0.46% a month. This formula contains only the standard deviation and the number of months; the shape of the distribution does not enter it. The fat-tailed market was built with the same standard deviations as the normal market, so every estimated mean in it is exactly as uncertain as under normal shocks: 0.46% a month in the example, under both distributions. Notebook 04 showed that the uncertainty in the means accounts for nearly all of the cost of estimation (3,087 of the 3,239 months of the critical window for 25 assets). The part of the problem that matters most is therefore untouched by the fat tails. In the proposition this is the case-1 term, N divided by M, which holds for any distribution with a finite variance; only the two covariance terms, k and h, were derived under normality.

The covariances. The sample covariance between two assets is the average, over the window, of the product of their two deviations from their means. A month in which both assets move five standard deviations contributes 25 times as much to that average as a month in which both move one standard deviation, so fat tails do make the sample covariance noisier. In this market, however, one scale factor multiplies every asset's shock in a given month. An extreme month therefore multiplies every entry of that month's contribution by the same number. This raises or lowers the whole estimated covariance matrix together and leaves the ratios between its entries almost unchanged, and the rules use only those ratios. The mean-variance weights, for example, are the inverse of the covariance matrix times the vector of means, rescaled so that the weights sum to one: if every entry of the covariance matrix is doubled, the product is halved and the rescaling undoes the halving, so the weights do not change. The minimum-variance weights behave the same way. For the constrained mean-variance rules the level of the covariance enters through the risk penalty, and the table shows that the effect is small there too. Fat tails of this kind therefore change the estimated level of risk and leave the estimated weights almost where they were, and the Sharpe ratio of a rule depends on its weights.

Together: the inputs that carry most of the cost, the means, are exactly as uncertain as under normal shocks, and the inputs that became noisier, the covariances, became noisier in a way the weights hardly respond to. The paper's conclusion therefore does not depend on the normality assumption.

What this notebook does not settle. Four limits remain.

First, the months in this market are still independent of one another: the volatility of one month says nothing about the next. In real markets volatility comes in waves, and turbulent months follow turbulent months. Under that pattern a covariance estimated from the last 120 months is wrong in a known direction for the month ahead, because the window averages over calm and stormy months while the next month belongs to whichever regime the market is in now. Notebook 06 builds a market with that pattern and repeats this notebook's three expectations on it.

Second, in this market the same scale factor multiplies every asset's shock in a given month, so an extreme month moves all assets together. A covariance matrix carries two kinds of information: the overall level of risk (how large the variances are) and the pattern of relative risk (which assets are more volatile than which others, and how strongly each pair moves together). A common scale changes the level and leaves the pattern almost intact, and the previous paragraph showed that the rules' weights depend only on the pattern. A market in which each asset had its own scale factor each month would change the pattern as well: in a month where one asset's scale is large and another's is small, their estimated variances and their estimated correlations with the other assets move apart. Fat tails of that kind would move the estimated weights, and the loss for the rules would be larger than the loss measured here. This notebook does not test that kind.

Third, the fat tails here are symmetric. The scale factor multiplies the shock whatever its sign, so an extreme month is as likely to be an extreme gain as an extreme loss. In real markets extreme losses are more common than extreme gains of the same size. A one-sided version would need a different construction and is not run.

Fourth, every number in this notebook comes from one draw of each market, that is, from one 24,000-month simulated history per distribution, all built from the same seed. A different seed would give slightly different Sharpe ratios for every rule with nothing else changed. Notebook 04 measured how much a Sharpe ratio moves between draws by building its market eight more times: for a rule with a Sharpe ratio near 0.13, the eight values spread over a range of about 0.01 to 0.02. That repetition was not done here, because it would cost eight further runs of the whole grid. A difference of about 0.005 between two cells of the tables above is therefore of the size that a change of seed alone produces, and it should not be read as an effect of the tails. The averages over many cells, and the counts in the three expectations, are the informative numbers.